## Student Management System - SQLite Project

This project demonstrates fundamental SQL operations in a Google Colab environment using an SQLite database. We'll create two related tables, `Students` and `Courses`, and perform various DDL, DML, query, and join operations.

In [ ]:
import sqlite3
import pandas as pd

# Connect to an SQLite database (it will be created if it doesn't exist)
conn = sqlite3.connect('student_management.db')
cursor = conn.cursor()

print('Connected to SQLite database: student_management.db')

## 1. DDL (Data Definition Language)

### CREATE TABLE

In [ ]:
# Drop tables if they already exist (useful for re-running the script for development)
cursor.execute('DROP TABLE IF EXISTS Students;')
cursor.execute('DROP TABLE IF EXISTS Courses;')
conn.commit()
print("Tables 'Students' and 'Courses' dropped (if they existed).")

# Create Courses table
cursor.execute('''
CREATE TABLE IF NOT EXISTS Courses (
    course_id INTEGER PRIMARY KEY AUTOINCREMENT,
    course_name TEXT NOT NULL UNIQUE,
    credits INTEGER
);
''')

# Create Students table with a foreign key to Courses
cursor.execute('''
CREATE TABLE IF NOT EXISTS Students (
    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    age INTEGER,
    enrollment_date DATE,
    course_id INTEGER,
    FOREIGN KEY (course_id) REFERENCES Courses(course_id)
);
''')

conn.commit()
print("Tables 'Courses' and 'Students' created successfully.")

### ALTER TABLE

In [ ]:
# Add a new column 'email' to the Students table
try:
    cursor.execute('ALTER TABLE Students ADD COLUMN email TEXT;')
    conn.commit()
    print("Column 'email' added to 'Students' table.")
except sqlite3.OperationalError as e:
    print(f"Could not add column 'email': {e} (might already exist).")

## 2. DML (Data Manipulation Language)

### INSERT

In [ ]:
# Insert data into Courses table
cursor.execute("INSERT INTO Courses (course_name, credits) VALUES ('Mathematics I', 3);")
cursor.execute("INSERT INTO Courses (course_name, credits) VALUES ('Computer Science Basics', 4);")
cursor.execute("INSERT INTO Courses (course_name, credits) VALUES ('History of Art', 3);")
cursor.execute("INSERT INTO Courses (course_name, credits) VALUES ('Physics II', 4);")
cursor.execute("INSERT INTO Courses (course_name, credits) VALUES ('Literature Survey', 3);")

# Insert data into Students table
cursor.execute("INSERT INTO Students (first_name, last_name, age, enrollment_date, course_id, email) VALUES ('Alice', 'Wonderland', 20, '2023-09-01', 1, 'alice.w@example.com');")
cursor.execute("INSERT INTO Students (first_name, last_name, age, enrollment_date, course_id, email) VALUES ('Bob', 'TheBuilder', 22, '2023-09-01', 2, 'bob.b@example.com');")
cursor.execute("INSERT INTO Students (first_name, last_name, age, enrollment_date, course_id, email) VALUES ('Charlie', 'Chaplin', 21, '2023-10-15', 1, 'charlie.c@example.com');")
cursor.execute("INSERT INTO Students (first_name, last_name, age, enrollment_date, course_id, email) VALUES ('Diana', 'Prince', 23, '2023-09-01', 3, 'diana.p@example.com');")
cursor.execute("INSERT INTO Students (first_name, last_name, age, enrollment_date, course_id, email) VALUES ('Eve', 'Adams', 19, '2023-11-01', 2, 'eve.a@example.com');")
cursor.execute("INSERT INTO Students (first_name, last_name, age, enrollment_date, course_id, email) VALUES ('Frank', 'Sinatra', 20, '2023-09-01', 4, 'frank.s@example.com');")

conn.commit()
print("Sample data inserted into 'Courses' and 'Students' tables.")

### UPDATE

In [ ]:
# Update a student's age and email
cursor.execute("UPDATE Students SET age = 21, email = 'alice.wonderland@example.com' WHERE student_id = 1;")
conn.commit()
print("Updated Alice Wonderland's age and email.")

# Display updated record
print("Alice Wonderland's updated record:")
cursor.execute("SELECT * FROM Students WHERE student_id = 1;")
display(pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description]))

### DELETE

In [ ]:
# Delete a student record (e.g., student with id 3 - Charlie Chaplin)
cursor.execute("DELETE FROM Students WHERE student_id = 3;")
conn.commit()
print("Deleted student with ID 3 (Charlie Chaplin).")

# Verify deletion by trying to select the deleted record
print("Attempting to retrieve student with ID 3 (should be empty):")
cursor.execute("SELECT * FROM Students WHERE student_id = 3;")
display(pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description]))

## 3. SQL Queries

### SELECT

In [ ]:
# Select all students
print("All Students:")
cursor.execute("SELECT * FROM Students;")
display(pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description]))

# Select all courses
print("\nAll Courses:")
cursor.execute("SELECT * FROM Courses;")
display(pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description]))

### WHERE

In [ ]:
# Select students enrolled in 'Computer Science Basics' (course_id = 2)
print("Students enrolled in Computer Science Basics:")
cursor.execute("SELECT first_name, last_name, age FROM Students WHERE course_id = 2;")
display(pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description]))

### ORDER BY

In [ ]:
# Select all students ordered by age in ascending order
print("Students ordered by age (ascending):")
cursor.execute("SELECT first_name, last_name, age FROM Students ORDER BY age ASC;")
display(pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description]))

### GROUP BY and COUNT

In [ ]:
# Count students per course
print("Number of students per course:")
cursor.execute('''
SELECT C.course_name, COUNT(S.student_id) AS num_students
FROM Courses C
LEFT JOIN Students S ON C.course_id = S.course_id
GROUP BY C.course_name;
''')
display(pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description]))

### GROUP BY and AVG

In [ ]:
# Calculate the average age of students per course
print("Average age of students per course:")
cursor.execute('''
SELECT C.course_name, AVG(S.age) AS average_age
FROM Courses C
LEFT JOIN Students S ON C.course_id = S.course_id
GROUP BY C.course_name;
''')
display(pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description]))

## 4. JOINS

### INNER JOIN

In [ ]:
# Retrieve students along with their enrolled course names
print("Students with their enrolled course names (INNER JOIN):")
cursor.execute('''
SELECT S.first_name, S.last_name, S.age, C.course_name, C.credits
FROM Students S
INNER JOIN Courses C ON S.course_id = C.course_id;
''')
display(pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description]))

## Close Database Connection

In [ ]:
# Close the database connection when done
conn.close()
print("Database connection closed.")